# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/div828/Flyrank_starternotebook/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### My baseline rule

For Lane 2 — Refresh / Content Opportunity Scoring, my baseline prioritises pages that already have meaningful historical search visibility and show unstable historical search position.

A page with more historical impressions has more existing search exposure at stake, while position instability provides an additional reason to review it.

### Reason codes

- `visible_position_unstable` — meaningful historical visibility and unstable historical search position.
- `visible_history` — meaningful historical visibility but no strong position instability.
- `below_review_threshold` — not enough historical visibility to prioritise for refresh review.

### Action labels

- `REVIEW_FOR_REFRESH` — prioritise for human refresh review.
- `MONITOR` — continue monitoring.

In [6]:
%pip -q install duckdb huggingface_hub

import os
import duckdb
import pandas as pd
import numpy as np

# Get Hugging Face token from Colab Secrets
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets first."

# Create DuckDB connection
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"""
read_parquet(
    '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("Connected to March 2026 warehouse.")

Connected to March 2026 warehouse.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Build page-level data for the March 2026 development window

pages = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(CASE
        WHEN report_date < DATE '2026-03-16'
        THEN gsc_impressions ELSE 0
    END) AS impressions_prev,

    SUM(CASE
        WHEN report_date >= DATE '2026-03-16'
        THEN gsc_impressions ELSE 0
    END) AS impressions_outcome,

    AVG(CASE
        WHEN report_date < DATE '2026-03-16'
        THEN gsc_avg_position
    END) AS avg_position_prev,

    STDDEV(CASE
        WHEN report_date < DATE '2026-03-16'
        THEN gsc_avg_position
    END) AS position_volatility_prev

FROM {FACT}
GROUP BY 1, 2
HAVING impressions_prev >= 100
""").df()

print("Pages created:", len(pages))
pages.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages created: 77540


,client_hash_id,content_hash_id,impressions_prev,impressions_outcome,avg_position_prev,position_volatility_prev
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,70.0,5.222776,2.612218
1,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,680.0,3.737399,2.554586
2,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,1614.0,6.156643,1.144948
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,246.0,83.0,4.520919,1.405275
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,413.0,359.0,4.390322,1.362661


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
from pathlib import Path

# Build baseline from the page-level data created earlier
baseline = pages.copy()

# Simple, transparent conditions
baseline["visible"] = baseline["impressions_prev"] >= 500
baseline["unstable"] = baseline["position_volatility_prev"] >= 5

# Baseline score
baseline["score"] = np.where(
    baseline["visible"],
    baseline["impressions_prev"] *
    (1 + baseline["unstable"].astype(int)),
    0
)

# ONE reason code per page
baseline["reason_code"] = np.select(
    [
        baseline["visible"] & baseline["unstable"],
        baseline["visible"]
    ],
    [
        "visible_position_unstable",
        "visible_history"
    ],
    default="below_review_threshold"
)

# Action
baseline["action_label"] = np.where(
    baseline["score"] > 0,
    "REVIEW_FOR_REFRESH",
    "MONITOR"
)

# Rank highest priority first
queue = baseline.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

# Write required CSV
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "baseline_action_score.csv"

queue[
    [
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action_label"
    ]
].to_csv(output_file, index=False)

print("Rows in ranked queue:", len(queue))
print("CSV written to:", output_file)

queue[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_prev",
        "position_volatility_prev",
        "score",
        "reason_code",
        "action_label"
    ]
].head(10)


Rows in ranked queue: 77540
CSV written to: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,impressions_prev,position_volatility_prev,score,reason_code,action_label
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83772.0,12.407646,167544.0,visible_position_unstable,REVIEW_FOR_REFRESH
1,client_e547b89c05043229,content_eadb33b5df496f4a,161575.0,0.151530,161575.0,visible_history,REVIEW_FOR_REFRESH
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,143173.0,1.161691,143173.0,visible_history,REVIEW_FOR_REFRESH
3,client_20259bd6705d81d4,content_82e35c4845e6c391,70169.0,7.539604,140338.0,visible_position_unstable,REVIEW_FOR_REFRESH
4,client_e547b89c05043229,content_ec2e0346994fb5a5,132811.0,0.387161,132811.0,visible_history,REVIEW_FOR_REFRESH
5,client_23a62021009f63c4,content_36e53e9c707674fc,109909.0,1.233209,109909.0,visible_history,REVIEW_FOR_REFRESH
6,client_62f4a7e64f5e0096,content_7172a7fad43f0998,108663.0,0.329356,108663.0,visible_history,REVIEW_FOR_REFRESH
7,client_62f4a7e64f5e0096,content_b99ea6861864dea5,91474.0,0.586987,91474.0,visible_history,REVIEW_FOR_REFRESH
8,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,87662.0,0.491442,87662.0,visible_history,REVIEW_FOR_REFRESH
9,client_62f4a7e64f5e0096,content_7c6373141eae744a,86860.0,0.937674,86860.0,visible_history,REVIEW_FOR_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Review the top 20 ranked pages

top20 = queue.head(20).copy()

top20["review_action"] = top20["action_label"]

top20["why_its_here"] = top20["reason_code"].map({
    "visible_position_unstable":
        "High historical search visibility with unstable historical position.",
    "visible_history":
        "Meaningful historical search visibility makes this page worth reviewing.",
    "below_review_threshold":
        "Below the current review threshold."
})

top20["what_would_make_it_wrong"] = np.where(
    top20["reason_code"] == "visible_position_unstable",
    "Position variation may be temporary and the content may still be current.",
    "Historical visibility alone may not mean the page actually needs a refresh."
)

review_cols = [
    "content_hash_id",
    "review_action",
    "reason_code",
    "why_its_here",
    "what_would_make_it_wrong"
]

top20[review_cols]


,content_hash_id,review_action,reason_code,why_its_here,what_would_make_it_wrong
0,content_9c057b66c30a3abb,REVIEW_FOR_REFRESH,visible_position_unstable,High historical search visibility with unstabl...,Position variation may be temporary and the co...
1,content_eadb33b5df496f4a,REVIEW_FOR_REFRESH,visible_history,Meaningful historical search visibility makes ...,Historical visibility alone may not mean the p...
2,content_e8a52cf3d5988c07,REVIEW_FOR_REFRESH,visible_history,Meaningful historical search visibility makes ...,Historical visibility alone may not mean the p...
3,content_82e35c4845e6c391,REVIEW_FOR_REFRESH,visible_position_unstable,High historical search visibility with unstabl...,Position variation may be temporary and the co...
4,content_ec2e0346994fb5a5,REVIEW_FOR_REFRESH,visible_history,Meaningful historical search visibility makes ...,Historical visibility alone may not mean the p...
5,content_36e53e9c707674fc,REVIEW_FOR_REFRESH,visible_history,Meaningful historical search visibility makes ...,Historical visibility alone may not mean the p...
6,content_7172a7fad43f0998,REVIEW_FOR_REFRESH,visible_history,Meaningful historical search visibility makes ...,Historical visibility alone may not mean the p...
7,content_b99ea6861864dea5,REVIEW_FOR_REFRESH,visible_history,Meaningful historical search visibility makes ...,Historical visibility alone may not mean the p...
8,content_e7b5dd4dff461ad2,REVIEW_FOR_REFRESH,visible_history,Meaningful historical search visibility makes ...,Historical visibility alone may not mean the p...
9,content_7c6373141eae744a,REVIEW_FOR_REFRESH,visible_history,Meaningful historical search visibility makes ...,Historical visibility alone may not mean the p...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Weak picks + leakage check

# Show some lower-confidence picks from the reviewed top 20
weak_picks = top20.sort_values(
    ["position_volatility_prev", "impressions_prev"],
    ascending=[True, True]
).head(5)

print("Potential weak picks:")
display(
    weak_picks[
        [
            "content_hash_id",
            "impressions_prev",
            "position_volatility_prev",
            "score",
            "reason_code",
            "action_label"
        ]
    ]
)

# Explicit leakage check
rule_inputs = [
    "impressions_prev",
    "position_volatility_prev"
]

forbidden_inputs = [
    "impressions_outcome",
    "is_declining"
]

leaked = [c for c in forbidden_inputs if c in rule_inputs]

print("\nRule inputs:", rule_inputs)
print("Future/label-derived inputs used:", leaked)

assert len(leaked) == 0, "Leakage detected in baseline rule!"

print("\nPASS — baseline score uses no future-window or label-derived inputs.")


Potential weak picks:


,content_hash_id,impressions_prev,position_volatility_prev,score,reason_code,action_label
1,content_eadb33b5df496f4a,161575.0,0.151530,161575.0,visible_history,REVIEW_FOR_REFRESH
6,content_7172a7fad43f0998,108663.0,0.329356,108663.0,visible_history,REVIEW_FOR_REFRESH
15,content_fd2117c2c6790e4b,78162.0,0.338162,78162.0,visible_history,REVIEW_FOR_REFRESH
4,content_ec2e0346994fb5a5,132811.0,0.387161,132811.0,visible_history,REVIEW_FOR_REFRESH
11,content_acbcc847f8996314,83715.0,0.389555,83715.0,visible_history,REVIEW_FOR_REFRESH



Rule inputs: ['impressions_prev', 'position_volatility_prev']
Future/label-derived inputs used: []

PASS — baseline score uses no future-window or label-derived inputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.